# Demo: Matrix Review + Left/Right Division for Solving Linear Systems

This notebook has two parts:

1. **A quick refresher** of the array/matrix concepts from the previous demo — just enough to warm back up, not a full re-teach.
2. **A new concept**: solving a linear system two different ways ("left division" and "right division"), and proving to yourself that both methods agree.

Follow along, run every demo cell, and complete the "Your Turn" exercises before moving on. None of the numbers below match Programming Project 3 — the goal is practicing the *methods*.


## Part 1: Quick Refresher

### Array Creation and Elementwise Arithmetic

Quick recap: `np.array()` builds an array from a list, and `+`, `-`, `*`, `/`, `**` on arrays are **elementwise** by default in NumPy.

In [1]:
a = np.array([2, 4, 6, 8])
print(a * 3)          # scalar multiplication -- every element scaled
print(a + 10)          # scalar addition -- every element shifted
print(a ** 2)          # elementwise squaring

[ 6 12 18 24]
[12 14 16 18]
[ 4 16 36 64]


### Your Turn

Given `v = np.array([1, 3, 5, 7])`, compute and print `(v + 2) ** 2` — every element, incremented by 2, then squared.

In [ ]:
# Your Turn
v = np.array([1, 3, 5, 7])
# TODO: compute and print (v + 2) ** 2


### 2D Arrays: Indexing and the `*` vs `@` Distinction

Recall the single most important matrix gotcha from last time: NumPy's `*` on two matrices is **elementwise**, while `@` is **true matrix multiplication**.

In [2]:
M = np.array([[1, 2], [3, 4]])
N = np.array([[5, 6], [7, 8]])

print(M[0, 1])   # single element: row 0, column 1
print(M * N)      # elementwise
print(M @ N)      # true matrix multiplication -- a DIFFERENT result

2
[[ 5 12]
 [21 32]]
[[19 22]
 [43 50]]


### The `@` Operator, In Depth

`@` is true matrix multiplication: the standard row-times-column linear algebra operation, not an elementwise one. The **dimension rule**: for `A @ B`, the number of *columns* in `A` must equal the number of *rows* in `B`. The result has the shape (rows of `A`, columns of `B`).

In [6]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])          # shape (2, 3)
B = np.array([[7, 8],
              [9, 10],
              [11, 12]])            # shape (3, 2)

C = A @ B
print("Shapes:", A.shape, "@", B.shape, "->", C.shape)
print(C)

Shapes: (2, 3) @ (3, 2) -> (2, 2)
[[ 58  64]
 [139 154]]


`@` also works between a matrix and a 1D vector — this is exactly matrix-vector multiplication, the same operation used to apply a linear system to a vector of unknowns:

In [7]:
v = np.array([1, 0, 1])   # length 3, matches A's 3 columns
result = A @ v
print(result)

[ 4 10]


If the dimension rule isn't satisfied, `@` raises a clear error rather than silently doing something unexpected:

In [8]:
try:
    A @ A   # A is (2,3) -- columns of the first A (3) don't match rows of the second A (2)
except ValueError as e:
    print("This failed, as expected:", e)

This failed, as expected: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)


### Your Turn — `@` Practice

Given `P = np.array([[2, 1], [0, 3], [4, 1]])` (shape 3x2) and `Q = np.array([[1, 2, 0], [3, 1, 2]])` (shape 2x3):

1. Predict the shape of `P @ Q` before running any code
2. Compute `P @ Q` and confirm your prediction
3. Try `Q @ P` as well — note that this is *also* valid (unlike the square-matrix case from before), but produces a **different-shaped** result. Print its shape.


In [ ]:
# Your Turn
P = np.array([[2, 1], [0, 3], [4, 1]])
Q = np.array([[1, 2, 0], [3, 1, 2]])

# TODO 1: print P @ Q and its shape

# TODO 2: print Q @ P and its shape -- compare to P @ Q


## Part 1.5: Transpose, In Depth

The transpose of a matrix flips it across its diagonal — every row becomes a column, and every column becomes a row. If `M` has shape `(rows, cols)`, then `M.T` has shape `(cols, rows)`.

In [9]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])   # shape (2, 3)

print("Original shape:", M.shape)
print(M)
print()
print("Transposed shape:", M.T.shape)
print(M.T)

Original shape: (2, 3)
[[1 2 3]
 [4 5 6]]

Transposed shape: (3, 2)
[[1 4]
 [2 5]
 [3 6]]


Transposing twice always gets you back to the original matrix — a useful fact for checking your work:

In [10]:
print(np.array_equal((M.T).T, M))   # transpose of a transpose = the original

True


### Transpose Combined with `@`: Dot Products and Outer Products

This is exactly the mechanism behind the right-division identity you'll use later in this notebook. A column vector transposed into a row, multiplied by the original column vector, computes a **sum of squares** (a 1x1 result). The reverse order produces a full outer-product matrix instead:

In [11]:
u = np.array([[1], [2], [3]])   # an explicit column vector, shape (3,1)

print(u.T @ u)   # (1,3) @ (3,1) -> shape (1,1): a single value, the sum of squares
print()
print(u @ u.T)   # (3,1) @ (1,3) -> shape (3,3): a full outer-product matrix

[[14]]

[[1 2 3]
 [2 4 6]
 [3 6 9]]


Notice this is the **same shape-flipping logic** used in the right-division formula from Part 2 below: transposing a vector or matrix changes which multiplication (`@`) is even valid, and often changes what the multiplication *means*.

### Your Turn — Transpose Practice

Given `N = np.array([[2, 4], [1, 3], [5, 0]])` (shape 3x2):

1. Print `N.T` and confirm its shape is `(2, 3)`
2. Compute `N.T @ N` and print its shape (this is a very common pattern in engineering/statistics — a matrix multiplied by its own transpose)
3. Compute `N @ N.T` and print its shape — confirm it's a **different** shape than `N.T @ N`


In [ ]:
# Your Turn
N = np.array([[2, 4], [1, 3], [5, 0]])

# TODO 1: print N.T and its shape

# TODO 2: print N.T @ N and its shape

# TODO 3: print N @ N.T and its shape


## Part 1.6: Matrix Inverse, In Depth

The inverse of a square matrix `A`, written $A^{-1}$, is the unique matrix such that $A \cdot A^{-1} = A^{-1} \cdot A = I$ (the identity matrix). Only **square** matrices can have an inverse, and even then, only if the matrix is non-singular (determinant not equal to 0, as covered in the earlier lecture).

In [12]:
A = np.array([[4, 7], [2, 6]])

A_inv = np.linalg.inv(A)
print("A inverse:")
print(A_inv)

A inverse:
[[ 0.6 -0.7]
 [-0.2  0.4]]


Verify the defining property directly — multiplying a matrix by its own inverse, in either order, gives the identity matrix:

In [13]:
print(np.round(A @ A_inv, 6))
print(np.round(A_inv @ A, 6))

[[ 1. -0.]
 [-0.  1.]]
[[ 1. -0.]
 [ 0.  1.]]


This works the same way for larger matrices:

In [14]:
A3 = np.array([[1, 2, 3],
               [0, 1, 4],
               [5, 6, 0]])

A3_inv = np.linalg.inv(A3)
print("A3 inverse:")
print(A3_inv)
print()
print("Check -- A3 @ A3_inv should be the identity matrix:")
print(np.round(A3 @ A3_inv, 6))

A3 inverse:
[[-24.  18.   5.]
 [ 20. -15.  -4.]
 [ -5.   4.   1.]]

Check -- A3 @ A3_inv should be the identity matrix:
[[ 1. -0.  0.]
 [ 0.  1.  0.]
 [ 0. -0.  1.]]


### Your Turn — Inverse Practice

Given `B = np.array([[3, 1], [2, 4]])`:

1. Compute `B_inv` using `np.linalg.inv()`
2. Verify `B @ B_inv` gives (approximately) the identity matrix, using `np.round(..., 6)` to clean up tiny floating-point noise
3. Compute `np.linalg.det(B)` — confirm it's nonzero (a nonzero determinant is *why* the inverse exists at all)


In [ ]:
# Your Turn
B = np.array([[3, 1], [2, 4]])

# TODO 1: compute B_inv

# TODO 2: verify B @ B_inv is (approximately) the identity matrix

# TODO 3: compute and print the determinant of B


**Reminder from the earlier lecture:** if a matrix is singular (determinant = 0), `np.linalg.inv()` raises a `LinAlgError` rather than returning a usable result — you'll practice this again at the very end of this notebook.

## Part 2: Left Division and Right Division

### The Problem: Solving $Ax = b$

Given a square matrix $A$ and a vector $b$, you often need to find the vector $x$ that satisfies $Ax = b$. MATLAB has two built-in operators for this:

- **Left division**, `A\b`, solves $Ax = b$ directly.
- **Right division**, `b/A`, solves a *differently arranged* system, $xA = b$ (x as a row vector, multiplied from the left).

Python has **no direct equivalent of either operator** — but both problems are still solvable in NumPy, and it's worth knowing both approaches because you'll see this exact left/right division pairing in MATLAB-derived course material.

### Method 1: Left Division — the Standard, Preferred Approach

For $Ax = b$, NumPy's direct, numerically stable tool is `np.linalg.solve(A, b)`. This is always your first choice when you just need to solve the system.

In [3]:
A = np.array([[3, 2], [1, 4]])
b = np.array([16, 14])

x_left = np.linalg.solve(A, b)
print("x (left division method):", x_left)

x (left division method): [3.6 2.6]


An equivalent (but less efficient and less numerically stable) way to get the same answer is computing the inverse explicitly:

In [4]:
x_left_v2 = np.linalg.inv(A) @ b
print("x (via explicit inverse):", x_left_v2)

x (via explicit inverse): [3.6 2.6]


### Method 2: Right Division — Solving the Same System a Different Way

There's a useful mathematical identity: solving $Ax = b$ is exactly equivalent to solving the *transposed*, row-vector version of the same system and transposing the answer back:

$$x = \left(b^T / A^T\right)^T$$

In NumPy, "right division" (`D/C`, solving $XC = D$) is computed as $X = D \cdot C^{-1}$. Applying that here, with $b$ playing the role of $D$ and $A^T$ playing the role of $C$:


In [5]:
x_right = b @ np.linalg.inv(A.T)
print("x (right division method):", x_right)

print("\nSame answer as left division?", np.allclose(x_left, x_right))

x (right division method): [3.6 2.6]

Same answer as left division? True


Both methods agree exactly. This is a genuinely useful sanity check: if you solve a system two independent ways and get the same answer, you can be much more confident it's correct — the same spirit as the multi-method verification you practiced in Programming Project 1.

### Your Turn

Given the 3x3 system below, solve for `x` using **both** methods and confirm they agree:

$$
\begin{bmatrix} 2 & 1 & -1 \\ 3 & -2 & 1 \\ 1 & 1 & 1 \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix}
=
\begin{bmatrix} 3 \\ -4 \\ 6 \end{bmatrix}
$$


In [ ]:
# Your Turn
A3 = np.array([[2, 1, -1],
               [3, -2, 1],
               [1, 1, 1]])
b3 = np.array([3, -4, 6])

# TODO 1: solve using the left division method (np.linalg.solve)

# TODO 2: solve using the right division method (b3 @ np.linalg.inv(A3.T))

# TODO 3: use np.allclose() to confirm both answers agree


### A Note on When You'd Actually Use Each Method

In practice, you'll almost always reach for `np.linalg.solve(A, b)` (left division) — it's simpler to write and more numerically reliable. The right-division approach above is worth knowing because:

1. It's the direct Python translation if you ever see MATLAB code using the `/` operator on a genuinely row-vector-shaped system (not just as an alternate way to solve a column-vector system, as practiced above)
2. It reinforces *why* `np.linalg.solve()` is preferred — computing two explicit matrix inverses (as the right-division method does) is more computationally expensive and less numerically stable than a single well-designed solve
3. Getting the same answer two genuinely different ways is a strong, fast correctness check on any linear system you solve — worth doing whenever the stakes are high enough to justify the extra few lines of code


### Your Turn — Sanity-Checking with `det()`

Before trusting *any* solve, it's worth confirming the matrix isn't singular (recall from the earlier lecture: a determinant of exactly 0 means no solution exists via this method).

Given `A4 = np.array([[4, 2], [8, 4]])`, compute its determinant with `np.linalg.det()`. What do you notice, and what would happen if you tried `np.linalg.solve()` on it?


In [ ]:
# Your Turn
A4 = np.array([[4, 2], [8, 4]])

# TODO: compute and print the determinant of A4

# TODO: try np.linalg.solve(A4, np.array([1, 2])) inside a try/except block
#       and print the error message if it fails
